### Trial-Exklusion und Datenaufbereitung

In diesem Abschnitt werden die Rohdaten aus dem Ordner `trialDataMainStudy` verarbeitet, bereinigt und für die statistische Analyse zusammengeführt.

#### Verarbeitungsschritte pro Versuchsperson:

1. **Behandlung der Übungsphase**: Die Daten der Übungsphase (`round 0`) werden markiert, jedoch nicht in die statistische Auswertung der Hauptphase einbezogen.
2. **Ausschlusskriterien der Hauptphase**: Innerhalb der Testrunden (`round > 0`) wird ein Trial als **ausgeschlossen** markiert (`trial_excl = 1`), wenn mindestens eine der folgenden Bedingungen zutrifft:
    * **Initialer Ausschluss**: Es handelt sich um den jeweils **ersten Trial** jeder Runde (`trial == 1`).
    * **Fehlversuch**: Das Ziel wurde nicht erfolgreich getroffen (`status == 0`).
    * **Zeitüberschreitung**: Die Reaktionszeit (`RT`) liegt über 4 Sekunden.
    * **Statistischer Ausreißer**: Die Reaktionszeit weicht um mehr als **drei Standardabweichungen (3 SD)** vom individuellen Mittelwert der Versuchsperson ab (berechnet auf Basis der gültigen Test-Trials).

Das Ergebnis wird als Übersicht in der Datei `trial_exclusion_overview.csv` gespeichert. Diese dient als Grundlage für die Berechnung der Exklusionsraten im Ergebnisteil der Arbeit.

In [1]:
import pandas as pd
import glob
import os

input_folder = "trialDataMainStudy"
output_file = "trial_exclusion_overview.csv"
summary_rows = []

files = glob.glob(os.path.join(input_folder, "*.csv"))
true_header = ["id","gameType","round","trial","timestamp","time","mouseX","mouseY",
               "position","effectDelay","startRT","endRT","RT","status","middleOrbExitCount"]

for file in files:
    df = pd.read_csv(file, names=true_header, header=0)
    
    test_data = df[df["round"] != 0].copy()
    clean_hits = test_data[(test_data["trial"] != 1) & (test_data["status"] == 1) & (test_data["RT"] < 4)]["RT"]
    m, sd = clean_hits.mean(), clean_hits.std()

    df["excl_round0"] = (df["round"] == 0).astype(int)
    df["excl_trial1"] = ((df["round"] != 0) & (df["trial"] == 1)).astype(int)
    df["excl_status0"] = ((df["round"] != 0) & (df["trial"] != 1) & (df["status"] == 0)).astype(int)
    df["excl_rt4000"] = ((df["round"] != 0) & (df["trial"] != 1) & (df["RT"] > 4)).astype(int)
    df["excl_3sd"] = ((df["round"] != 0) & (df["trial"] != 1) & (df["status"] == 1) & (df["RT"] <= 4) & \
                     ((df["RT"] > (m + 3*sd)) | (df["RT"] < (m - 3*sd)))).astype(int)

    df["trial_excl"] = (df["excl_round0"] | df["excl_trial1"] | df["excl_status0"] | df["excl_rt4000"] | df["excl_3sd"])

    summary = df[["id", "round", "trial", "excl_trial1", "excl_status0", "excl_rt4000", "excl_3sd", "trial_excl"]]
    summary_rows.append(summary)

if summary_rows:
    final_df = pd.concat(summary_rows, ignore_index=True)
    final_df.to_csv(output_file, index=False)
    
    test_phase = final_df[final_df["round"] != 0]
    n_test = len(test_phase)
    
    print(f"--- Statistik der Testphase (N = {n_test} Trials) ---")
    print(f"Erster Trial: {test_phase['excl_trial1'].sum() / n_test * 100:.2f} %")
    print(f"Status 0: {test_phase['excl_status0'].sum() / n_test * 100:.2f} %")
    print(f"RT > 4000ms: {test_phase['excl_rt4000'].sum() / n_test * 100:.2f} %")
    print(f"3-SD Ausreißer: {test_phase['excl_3sd'].sum() / n_test * 100:.2f} %")
    print("-" * 30)
    print(f"Gesamt-Exklusion: {test_phase['trial_excl'].sum() / n_test * 100:.2f} %")

--- Statistik der Testphase (N = 20640 Trials) ---
Erster Trial: 1.67 %
Status 0: 7.85 %
RT > 4000ms: 0.00 %
3-SD Ausreißer: 0.82 %
------------------------------
Gesamt-Exklusion: 10.34 %


In [2]:
import pandas as pd
import glob
import os

input_folder = "trialDataMainStudy"
output_file = "trial_exclusion_overview.csv"
summary_rows = []

files = glob.glob(os.path.join(input_folder, "*.csv"))
true_header = ["id","gameType","round","trial","timestamp","time","mouseX","mouseY",
               "position","effectDelay","startRT","endRT","RT","status","middleOrbExitCount"]

for file in files:
    df = pd.read_csv(file, names=true_header, header=0)
    df["middleOrbExitCount"] = pd.to_numeric(df["middleOrbExitCount"].astype(str).str.replace(';', ''), errors='coerce').fillna(0).astype(int)
    test_data = df[df["round"] != 0].copy()
    clean_hits = test_data[(test_data["trial"] != 1) & (test_data["status"] == 1) & (test_data["RT"] < 4)]["RT"]
    m, sd = clean_hits.mean(), clean_hits.std()

    df["excl_round0"] = (df["round"] == 0).astype(int)
    df["excl_trial1"] = ((df["round"] != 0) & (df["trial"] == 1)).astype(int)
    df["excl_status0"] = ((df["round"] != 0) & (df["trial"] != 1) & (df["status"] == 0)).astype(int)
    df["excl_rt4000"] = ((df["round"] != 0) & (df["trial"] != 1) & (df["RT"] > 4)).astype(int)
    
    df["excl_orbExit"] = ((df["round"] != 0) & (df["trial"] != 1) & (df["middleOrbExitCount"] > 1)).astype(int)
    
    df["excl_3sd"] = ((df["round"] != 0) & (df["trial"] != 1) & (df["status"] == 1) & (df["RT"] <= 4) & \
                     ((df["RT"] > (m + 3*sd)) | (df["RT"] < (m - 3*sd)))).astype(int)

    df["trial_excl"] = (df["excl_round0"] | df["excl_trial1"] | df["excl_status0"] | 
                        df["excl_rt4000"] | df["excl_orbExit"] | df["excl_3sd"])

    summary = df[["id", "round", "trial", "excl_trial1", "excl_status0", "excl_rt4000", "excl_orbExit", "excl_3sd", "trial_excl"]]
    summary_rows.append(summary)

if summary_rows:
    final_df = pd.concat(summary_rows, ignore_index=True)
    # final_df.to_csv(output_file, index=False)
    
    test_phase = final_df[final_df["round"] != 0]
    n_test = len(test_phase)
    
    print(f"--- Statistik der Testphase (N = {n_test} Trials) ---")
    print(f"Erster Trial: {test_phase['excl_trial1'].sum() / n_test * 100:.2f} %")
    print(f"Status 0: {test_phase['excl_status0'].sum() / n_test * 100:.2f} %")
    print(f"RT > 4000ms: {test_phase['excl_rt4000'].sum() / n_test * 100:.2f} %")
    print(f"Orb Exits > 1: {test_phase['excl_orbExit'].sum() / n_test * 100:.2f} %")
    print(f"3-SD Ausreißer: {test_phase['excl_3sd'].sum() / n_test * 100:.2f} %")
    print("-" * 30)
    print(f"Gesamt-Exklusion: {test_phase['trial_excl'].sum() / n_test * 100:.2f} %")

--- Statistik der Testphase (N = 20640 Trials) ---
Erster Trial: 1.67 %
Status 0: 7.85 %
RT > 4000ms: 0.00 %
Orb Exits > 1: 1.98 %
3-SD Ausreißer: 0.82 %
------------------------------
Gesamt-Exklusion: 12.10 %
